# Static UMAP Plotting with Cross-Matched Sample Overlays

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from pathlib import Path
import logging
import hyrax

#import nbimporter
import static_umap_plotting as sup

In [ ]:
def load_external_catalog(catalog_path: str) -> pd.DataFrame:
    """Load an overlay catalog from parquet or FITS."""
    catalog_path = Path(catalog_path)
    suffix = catalog_path.suffix.lower()

    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(catalog_path)

    if suffix in {".fits", ".fit", ".fts"}:
        from astropy.table import Table

        return Table.read(catalog_path).to_pandas()

    raise ValueError(f"Unsupported catalog format '{suffix}'. Use parquet or FITS.")


def _resolve_catalog_id_column(catalog: pd.DataFrame, catalog_id_column: str = None) -> str:
    """Pick the catalog object-ID column, preferring user input when provided."""
    candidates = []
    if catalog_id_column is not None:
        candidates.append(catalog_id_column)

    candidates.extend([
        'object_id',
        'rubin_object_id',
        'objectId',
        'objectId_data',
        'id',
    ])

    for candidate in candidates:
        if candidate in catalog.columns:
            return candidate

    raise KeyError(
        "Could not find an object ID column in catalog. "
        "Pass catalog_id_column explicitly."
    )


def _normalize_object_ids(values) -> pd.Series:
    """Normalize object IDs to comparable strings while preserving missing values."""
    ids = pd.Series(values, copy=False)
    ids = ids.map(
        lambda value: value.decode('utf-8')
        if isinstance(value, (bytes, bytearray))
        else value
    )

    missing = ids.isna()
    numeric = pd.to_numeric(ids, errors='coerce')

    if (~missing).any() and numeric.loc[~missing].notna().all():
        normalized = numeric.astype('Int64').astype('string')
    else:
        normalized = ids.astype('string').str.strip()

    return normalized.mask(missing)


In [ ]:
def get_umap_with_ids(config=None, input_dir=None, suppress_logs=True, id_field: str = 'objectId_data'):
    """
    Load UMAP results and return coordinates with object IDs.

    Parameters
    ----------
    config : object
        Hyrax config object.
    input_dir : str or Path, optional
        Directory containing UMAP results.
    suppress_logs : bool, default True
        If True, suppress hyrax logging output.
    id_field : str, default 'objectId_data'
        Preferred metadata field containing object IDs.

    Returns
    -------
    dict
        Dictionary with keys:
        - 'x': array of UMAP x coordinates
        - 'y': array of UMAP y coordinates
        - 'rubin_ids': array of object IDs
        - 'id_field': metadata field used for IDs
        - 'umap_results': the InferenceDataSet object
    """
    from hyrax.data_sets.inference_dataset import InferenceDataSet

    def _extract_metadata_column(metadata_obj, field_name: str):
        """Extract one column from different metadata return types."""
        if isinstance(metadata_obj, dict):
            if field_name in metadata_obj:
                return np.asarray(metadata_obj[field_name])
            if len(metadata_obj) == 1:
                return np.asarray(next(iter(metadata_obj.values())))
            return None

        if hasattr(metadata_obj, 'columns'):
            columns = list(metadata_obj.columns)
            if field_name in columns:
                return metadata_obj[field_name].to_numpy()
            if len(columns) == 1:
                return metadata_obj[columns[0]].to_numpy()
            return None

        dtype = getattr(metadata_obj, 'dtype', None)
        names = getattr(dtype, 'names', None)
        if names:
            if field_name in names:
                return np.asarray(metadata_obj[field_name])
            if len(names) == 1:
                return np.asarray(metadata_obj[names[0]])
            return None

        arr = np.asarray(metadata_obj)
        if arr.ndim == 1:
            return arr
        if arr.ndim == 2 and arr.shape[1] == 1:
            return arr[:, 0]
        return None

    if suppress_logs:
        logging.disable(logging.CRITICAL)

    umap_results = InferenceDataSet(config, results_dir=input_dir, verb="umap")

    logging.disable(logging.NOTSET)

    # Extract 2D coordinates
    points = np.array([point.numpy() for point in umap_results])
    x, y = points[:, 0], points[:, 1]

    all_indices = list(range(len(umap_results)))
    available_fields = list(umap_results.metadata_fields())

    preferred_fields = [
        id_field,
        'objectId_data',
        'object_id_data',
        'objectId',
        'object_id',
        'rubin_object_id',
        'id',
    ]

    # Deduplicate while preserving order
    deduped_fields = []
    seen = set()
    for field in preferred_fields:
        if field is None or field in seen:
            continue
        seen.add(field)
        deduped_fields.append(field)

    # Try fields listed by metadata_fields() first, then the rest
    candidate_fields = [f for f in deduped_fields if f in available_fields] + [
        f for f in deduped_fields if f not in available_fields
    ]

    rubin_ids = None
    resolved_field = None
    attempts = []

    for candidate in candidate_fields:
        try:
            metadata = umap_results.metadata(all_indices, [candidate])
            extracted = _extract_metadata_column(metadata, candidate)

            if extracted is None:
                attempts.append(f"{candidate}: field not found in metadata payload")
                continue

            if len(extracted) != len(umap_results):
                attempts.append(
                    f"{candidate}: length mismatch ({len(extracted)} vs {len(umap_results)})"
                )
                continue

            rubin_ids = np.asarray(extracted)
            resolved_field = candidate
            break
        except Exception as exc:
            attempts.append(f"{candidate}: {exc}")

    if rubin_ids is None:
        raise ValueError(
            "Could not extract object IDs from UMAP metadata. "
            f"Available metadata fields: {available_fields}. "
            f"Attempts: {' | '.join(attempts)}"
        )

    return {
        'x': x,
        'y': y,
        'rubin_ids': rubin_ids,
        'id_field': resolved_field,
        'umap_results': umap_results,
    }



In [ ]:
def match_catalog_to_umap(
    umap_data: dict,
    catalog: pd.DataFrame,
    key: str,
    threshold: float = 0.0,
    catalog_id_column: str = None,
) -> pd.DataFrame:
    """Return catalog rows that pass `key >= threshold` and match UMAP object IDs."""
    if key not in catalog.columns:
        raise KeyError(f"Catalog column '{key}' not found. Available columns: {list(catalog.columns)}")

    catalog_id_column = _resolve_catalog_id_column(catalog, catalog_id_column)
    values = pd.to_numeric(catalog[key], errors='coerce')
    selected = catalog.loc[values >= threshold, [catalog_id_column]].copy()
    selected[key] = values.loc[selected.index]

    if selected.empty:
        return pd.DataFrame(columns=['x', 'y', key])

    umap_lookup = pd.DataFrame({
        'x': umap_data['x'],
        'y': umap_data['y'],
        '_match_id': _normalize_object_ids(umap_data['rubin_ids']),
    })

    selected['_match_id'] = _normalize_object_ids(selected[catalog_id_column])
    selected = selected.loc[selected['_match_id'].notna()]

    return selected.merge(umap_lookup, on='_match_id', how='inner')


def plot_umap_flag_overlay(
    ax,
    umap_data: dict,
    catalog: pd.DataFrame,
    flag: dict,
    catalog_id_column: str = None,
    alpha_background: float = 0.5,
    s_background: float = 1,
    title: str = None,
    show_legend: bool = True,
) -> pd.DataFrame:
    """Plot one UMAP panel with one merger-flag overlay."""
    key = flag['key']
    threshold = flag.get('threshold', 0.0)
    color = flag.get('color', 'tab:red')
    marker = flag.get('marker', 'x')
    label = flag.get('label', key)
    alpha = flag.get('alpha', 0.6)
    size = flag.get('s', 5)

    ax.scatter(
        umap_data['x'],
        umap_data['y'],
        alpha=alpha_background,
        s=s_background,
        c='gray',
        label='All',
    )

    matched = match_catalog_to_umap(
        umap_data,
        catalog,
        key,
        threshold=threshold,
        catalog_id_column=catalog_id_column,
    )

    if not matched.empty:
        ax.scatter(
            matched['x'].to_numpy(),
            matched['y'].to_numpy(),
            alpha=alpha,
            s=size,
            c=color,
            marker=marker,
            label=f"{label} (n={len(matched)})",
        )

    if title:
        ax.set_title(title)
    if show_legend:
        ax.legend(loc='best', fontsize='small')

    return matched


In [ ]:
def load_umap_for_run_expt(run: int, expt: int, suppress_logs: bool = True) -> dict:
    """Load UMAP coordinates and object IDs for one run/expt pair."""
    umap_dir, config_file = sup.extract_umap_info(run, expt)

    if suppress_logs:
        logging.disable(logging.CRITICAL)
    h = hyrax.Hyrax(config_file=config_file)
    logging.disable(logging.NOTSET)

    return get_umap_with_ids(
        config=h.config,
        input_dir=umap_dir,
        suppress_logs=suppress_logs,
    )


def plot_umap_merger_flag_grid(
    run: int,
    expts,
    catalog: pd.DataFrame,
    flags: list,
    catalog_id_column: str = None,
    figsize: tuple = None,
    dpi: int = 150,
    save_path: str = None,
    suptitle: str = None,
    suppress_logs: bool = True,
    alpha_background: float = 0.5,
    s_background: float = 1,
    show_legend: bool = True,
    suptitle_y: float = 0.995,
    top_margin: float = 0.965,
):
    """Plot one row per experiment and one column per merger flag."""
    from tqdm.notebook import tqdm

    expts = list(expts)
    nrows = len(expts)
    ncols = len(flags)

    if nrows == 0:
        raise ValueError("At least one experiment is required.")
    if ncols == 0:
        raise ValueError("At least one flag configuration is required.")
    if figsize is None:
        figsize = (ncols * 4, nrows * 3)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        dpi=dpi,
        squeeze=False,
    )

    for row, expt in enumerate(tqdm(expts, total=nrows)):
        try:
            umap_data = load_umap_for_run_expt(
                run,
                expt,
                suppress_logs=suppress_logs,
            )
        except Exception as exc:
            for col in range(ncols):
                ax = axes[row, col]
                ax.text(
                    0.5,
                    0.5,
                    f"Error loading\nRun {run}, Expt {expt}\n{exc}",
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                )
                ax.set_title(f"Run {run}, Expt {expt}")
            continue

        for col, flag in enumerate(flags):
            ax = axes[row, col]
            label = flag.get('label', flag['key'])
            title = f"Run {run}, Expt {expt}: {label}"

            try:
                plot_umap_flag_overlay(
                    ax,
                    umap_data,
                    catalog,
                    flag,
                    catalog_id_column=catalog_id_column,
                    alpha_background=alpha_background,
                    s_background=s_background,
                    title=title,
                    show_legend=show_legend,
                )
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    str(exc),
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                )
                ax.set_title(title)

    if suptitle:
        fig.suptitle(suptitle, fontsize=16, y=suptitle_y)

    fig.tight_layout(rect=(0, 0, 1, top_margin if suptitle else 1))

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
    else:
        plt.show()

    return fig, axes


In [ ]:
MERGER_FLAGS = [
    {
        'key': 'has_mini_past_1gyr',
        'threshold': 0.5,
        'color': 'blue',
        'marker': '.',
        'label': 'Mini',
    },
    {
        'key': 'has_minor_past_1gyr',
        'threshold': 0.5,
        'color': 'green',
        'marker': 's',
        'label': 'Minor',
    },
    {
        'key': 'has_major_past_1gyr',
        'threshold': 0.5,
        'color': 'red',
        'marker': 'x',
        'label': 'Major',
    },
]


CATALOG_PATHS = {
    'all': "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog.fits",
    'le_120x120': "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_le_120x120.fits",
    'gt_120x120': "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_gt_120x120.fits",
}


In [ ]:
RUN_CATALOGS = {
    2: 'all',
    3: 'le_120x120',
    4: 'gt_120x120',
    5: 'all',
    6: 'le_120x120',
    7: 'gt_120x120',
    8: 'all',
}


def plot_run_merger_flags(
    run: int,
    expts=range(1, 9),
    flags=MERGER_FLAGS,
    catalog_key: str = None,
    **kwargs,
):
    """Load the catalog for `run` and plot expt rows by merger-flag columns."""
    resolved_catalog_key = catalog_key or RUN_CATALOGS[run]
    catalog = load_external_catalog(CATALOG_PATHS[resolved_catalog_key])

    return plot_umap_merger_flag_grid(
        run=run,
        expts=expts,
        catalog=catalog,
        flags=flags,
        suptitle=f"Run {run} ({resolved_catalog_key})",
        **kwargs,
    )


## Example Usage

Set `run`, `expts`, and `flags` to control the grid. The default grid is three columns wide (`mini`, `minor`, `major`) and one row per experiment.


### Run 2


In [ ]:
fig, axes = plot_run_merger_flags(
    run=2,
    expts=range(1, 9),
    alpha_background=0.5,
    show_legend=True,
)


### Optional Run Batch


In [ ]:
# Uncomment to generate grids for several runs.
# for run in [2, 3, 4, 5, 6, 7, 8]:
#     fig, axes = plot_run_merger_flags(
#         run=run,
#         expts=range(1, 9),
#         alpha_background=0.5,
#         show_legend=True,
#     )
